# 05 — Threshold, evaluación limpia y robustez

Orden obligatorio: (1) calibrar el threshold solo con validation, (2) congelarlo y evaluar test limpio una vez, (3) ejecutar stress tests por separado.

```bash
python -m src.evaluation.evaluate --mode calibrate --experiment-name baseline_formal/baseline_con_aumento --batch-size 32 --criterion max_f1
python -m src.evaluation.evaluate --mode test --experiment-name baseline_formal/baseline_con_aumento --batch-size 32
python -m src.evaluation.evaluate_stress --experiment-name baseline_formal/baseline_con_aumento --batch-size 32 --seed 2026
```

El criterio maximiza F1 en validation; si hay empate, prioriza menor FAR y luego menor FRR. Ningún resultado de test interviene en esa selección.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPERIMENT_DIR = PROJECT_ROOT / 'outputs/experiments/baseline_formal/baseline_con_aumento'

for filename in ('threshold.json', 'validation_metrics.json', 'test_metrics.json'):
    path = EXPERIMENT_DIR / filename
    if path.exists():
        print(f'\n{filename}')
        display(json.loads(path.read_text(encoding='utf-8')))

In [ ]:
stress_path = EXPERIMENT_DIR / 'stress_metrics.csv'
if stress_path.exists():
    display(pd.read_csv(stress_path))
for filename in ('training_history.png', 'validation_roc_curve.png', 'validation_precision_recall_curve.png', 'test_confusion_matrix.png', 'stress_summary.png'):
    path = EXPERIMENT_DIR / filename
    if path.exists():
        display(Image(filename=str(path)))

Las métricas antiguas no se consideran finales porque provenían del protocolo previo con fuga. Las métricas de esta carpeta corresponden a splits por video auditados sin fuga. Las alteraciones de estrés afectan solo `image_b` (probe) y se reportan fuera del test limpio.